# 1D CNN & 2D CNN — 텍스트 분류

## 개념 비교

| 구분 | 2D CNN (이미지) | 1D CNN (텍스트) |
|------|----------------|----------------|
| **커널 형태** | `(H_k, W_k)` | `(kernel_size,)` |
| **감지 패턴** | 공간적 패턴 (엣지, 모서리) | 시간적 패턴 (n-gram) |
| **입력 형태** | `(B, C, H, W)` | `(B, E, L)` |
| **슬라이딩 방향** | 가로 + 세로 | 시퀀스 방향만 |

## TextCNN 구조 (Kim, 2014)

```
Embedding
    │
    ├── Conv1d(k=2) → ReLU → GlobalMaxPool
    ├── Conv1d(k=3) → ReLU → GlobalMaxPool   ← 각기 다른 n-gram 패턴
    └── Conv1d(k=4) → ReLU → GlobalMaxPool
              │
           Concat → Dropout → Linear → Softmax
```

> **핵심**: 각 필터 = 특정 n-gram 탐지기 / Global Max Pooling = 위치 불변성

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
from collections import Counter

torch.manual_seed(42); random.seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cpu


## 1️⃣ 1D vs 2D CNN 개념 시연

In [2]:
print('=' * 55)
print('  CNN 개념 비교: 1D vs 2D')
print('=' * 55)

# 2D Conv — 이미지
print('\n[2D CNN — 이미지 처리]')
img = torch.randn(2, 1, 8, 8)   # (B, C, H, W)
c2d = nn.Conv2d(1, 4, kernel_size=3, padding=1)
o2d = c2d(img)
print(f'  입력: {tuple(img.shape)}  → (배치, 채널, 높이, 너비)')
print(f'  커널: (4, 1, 3, 3)   → 4개 필터, 3×3')
print(f'  출력: {tuple(o2d.shape)}')

# 1D Conv — 텍스트
print('\n[1D CNN — 텍스트 처리]')
seq = torch.randn(2, 16, 10)    # (B, E, L)
c1d = nn.Conv1d(16, 4, kernel_size=3, padding=1)
o1d = c1d(seq)
print(f'  입력: {tuple(seq.shape)}  → (배치, 임베딩, 시퀀스 길이)')
print(f'  커널: (4, 16, 3)   → 4개 필터, 3-gram')
print(f'  출력: {tuple(o1d.shape)}')

pooled = F.max_pool1d(o1d, kernel_size=o1d.size(-1))
print(f'  MaxPool: {tuple(pooled.shape)}  ← 위치 불변 최대 활성화')

# 슬라이딩 직관
print('\n[1D Conv 슬라이딩 직관]')
sentence = ['I', 'love', 'this', 'movie', 'very', 'much']
print(f'  문장: {sentence}')
print(f'  kernel_size=3 필터가 보는 trigram:')
for i in range(len(sentence) - 2):
    print(f'    위치 {i}: {sentence[i:i+3]}')

  CNN 개념 비교: 1D vs 2D

[2D CNN — 이미지 처리]
  입력: (2, 1, 8, 8)  → (배치, 채널, 높이, 너비)
  커널: (4, 1, 3, 3)   → 4개 필터, 3×3
  출력: (2, 4, 8, 8)

[1D CNN — 텍스트 처리]
  입력: (2, 16, 10)  → (배치, 임베딩, 시퀀스 길이)
  커널: (4, 16, 3)   → 4개 필터, 3-gram
  출력: (2, 4, 10)
  MaxPool: (2, 4, 1)  ← 위치 불변 최대 활성화

[1D Conv 슬라이딩 직관]
  문장: ['I', 'love', 'this', 'movie', 'very', 'much']
  kernel_size=3 필터가 보는 trigram:
    위치 0: ['I', 'love', 'this']
    위치 1: ['love', 'this', 'movie']
    위치 2: ['this', 'movie', 'very']
    위치 3: ['movie', 'very', 'much']


## 2️⃣ 데이터 준비 (감성 분류)

In [3]:
SENTIMENT_DATA = [
    ('this movie is absolutely fantastic and amazing',     1),
    ('great film with wonderful acting performances',      1),
    ('i really loved every moment of this masterpiece',   1),
    ('brilliant story and excellent cinematography',       1),
    ('best movie i have ever seen in my life',             1),
    ('incredible plot with outstanding performances',      1),
    ('heartwarming and beautifully crafted film',          1),
    ('superb direction and captivating storyline',         1),
    ('thoroughly enjoyed this delightful experience',      1),
    ('amazing characters and wonderful dialogue',          1),
    ('this film exceeded all my expectations perfectly',   1),
    ('a true cinematic gem worth watching again',          1),
    ('terrible movie with poor acting and bad script',     0),
    ('worst film i have ever seen so boring',              0),
    ('completely disappointing and waste of time',         0),
    ('awful story with dull characters',                   0),
    ('horrible experience nothing made sense',             0),
    ('very bad movie do not recommend to anyone',          0),
    ('boring and predictable with no redeeming quality',   0),
    ('terrible direction and extremely poor writing',      0),
    ('dull plot and annoying characters throughout',       0),
    ('absolutely dreadful cinema not worth your time',     0),
    ('poorly made with ridiculous storyline',              0),
    ('dreadful acting and incoherent mess of a film',      0),
]

class SimpleVocab:
    def __init__(self):
        self.w2i = {'<PAD>': 0, '<UNK>': 1}
    def build(self, texts):
        for text in texts:
            for w in text.lower().split():
                if w not in self.w2i:
                    self.w2i[w] = len(self.w2i)
        print(f'[Vocab] 크기: {len(self.w2i)}')
    def encode(self, text, max_len=20):
        ids = [self.w2i.get(w, 1) for w in text.lower().split()]
        return (ids + [0]*max_len)[:max_len]
    def __len__(self): return len(self.w2i)


class SentimentDataset(Dataset):
    def __init__(self, data, vocab, max_len=20):
        self.samples = [(torch.tensor(vocab.encode(t, max_len)), torch.tensor(l))
                        for t, l in data]
    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


random.shuffle(SENTIMENT_DATA)
split = int(len(SENTIMENT_DATA) * 0.75)
train_data, test_data = SENTIMENT_DATA[:split], SENTIMENT_DATA[split:]

vocab = SimpleVocab()
vocab.build([t for t, _ in SENTIMENT_DATA])

MAX_LEN = 20
train_loader = DataLoader(SentimentDataset(train_data, vocab, MAX_LEN), batch_size=4, shuffle=True)
test_loader  = DataLoader(SentimentDataset(test_data,  vocab, MAX_LEN), batch_size=4)
print(f'Train: {len(train_data)}  /  Test: {len(test_data)}')

[Vocab] 크기: 98
Train: 18  /  Test: 6


## 3️⃣ 모델 ① — 단순 1D CNN

In [4]:
class SimpleCNN1D(nn.Module):
    """단일 kernel_size의 1D CNN 분류기"""
    def __init__(self, vocab_size, embed_dim=32, num_filters=32,
                 kernel_size=3, num_classes=2, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv      = nn.Conv1d(embed_dim, num_filters, kernel_size, padding=kernel_size//2)
        self.dropout   = nn.Dropout(0.3)
        self.fc        = nn.Linear(num_filters, num_classes)

    def forward(self, x):
        emb  = self.embedding(x).permute(0, 2, 1)           # (B, E, L)
        feat = F.relu(self.conv(emb))                        # (B, F, L)
        pool = F.max_pool1d(feat, feat.size(-1)).squeeze(-1) # (B, F)
        return self.fc(self.dropout(pool))

m = SimpleCNN1D(len(vocab))
print(m)
x_test = torch.randint(0, len(vocab), (2, MAX_LEN))
print(f'출력 shape: {m(x_test).shape}')

SimpleCNN1D(
  (embedding): Embedding(98, 32, padding_idx=0)
  (conv): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=32, out_features=2, bias=True)
)
출력 shape: torch.Size([2, 2])


## 4️⃣ 모델 ② — TextCNN (멀티스케일, Kim 2014)

In [5]:
class TextCNN(nn.Module):
    """
    Kim (2014) 'Convolutional Neural Networks for Sentence Classification'
    여러 kernel_size를 병렬 적용 → 다양한 n-gram 패턴 동시 포착
    """
    def __init__(self, vocab_size, embed_dim=64, num_filters=64,
                 kernel_sizes=(2, 3, 4), num_classes=2,
                 dropout=0.5, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        # ★ 각 kernel_size마다 독립적인 Conv 레이어
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)
        self._init()

    def _init(self):
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        for c in self.convs:
            nn.init.kaiming_uniform_(c.weight)
            nn.init.zeros_(c.bias)

    def forward(self, x):
        emb    = self.embedding(x).permute(0, 2, 1)          # (B, E, L)
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(emb))                            # (B, F, L')
            p = F.max_pool1d(c, c.size(-1)).squeeze(-1)      # (B, F)
            pooled.append(p)
        cat = torch.cat(pooled, dim=1)                       # (B, F*num_kernels)
        return self.fc(self.dropout(cat))

## 5️⃣ 모델 ③ — TextCNN 2D (2D Conv 적용 방식)

In [6]:
class TextCNN2D(nn.Module):
    """
    2D CNN을 텍스트에 적용:
    입력을 (1, seq_len, embed_dim) 이미지로 간주
    커널: (k, embed_dim) → k개 단어의 임베딩 전체 수용
    """
    def __init__(self, vocab_size, embed_dim=64, num_filters=32,
                 kernel_heights=(2, 3, 4), num_classes=2,
                 dropout=0.5, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (kh, embed_dim)) for kh in kernel_heights
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_heights), num_classes)

    def forward(self, x):
        emb = self.embedding(x).unsqueeze(1)         # (B, 1, L, E)
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(emb)).squeeze(3)         # (B, F, L')
            p = F.max_pool1d(c, c.size(-1)).squeeze(-1)
            pooled.append(p)
        return self.fc(self.dropout(torch.cat(pooled, 1)))

print('모델 구조 확인')
for name, cls, kwargs in [
    ('Simple CNN1D',  SimpleCNN1D,  dict(vocab_size=len(vocab))),
    ('TextCNN 1D',    TextCNN,      dict(vocab_size=len(vocab))),
    ('TextCNN 2D',    TextCNN2D,    dict(vocab_size=len(vocab))),
]:
    m = cls(**kwargs)
    n = sum(p.numel() for p in m.parameters())
    print(f'  {name:<16}: 파라미터 {n:,}')

모델 구조 확인
  Simple CNN1D    : 파라미터 6,306
  TextCNN 1D      : 파라미터 43,714
  TextCNN 2D      : 파라미터 24,994


## 6️⃣ 학습 & 비교 실험

In [7]:
def train_and_eval(model, train_loader, test_loader, epochs=40, lr=1e-3):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    best_acc  = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
                total   += y.size(0)
        best_acc = max(best_acc, correct / total)

    return best_acc, model


models = {
    'Simple CNN1D (k=3)'   : SimpleCNN1D(len(vocab), embed_dim=32, num_filters=32, kernel_size=3),
    'TextCNN 1D (k=2,3,4)' : TextCNN(len(vocab),    embed_dim=64, num_filters=32, kernel_sizes=(2,3,4)),
    'TextCNN 2D (k=2,3,4)' : TextCNN2D(len(vocab),  embed_dim=64, num_filters=32, kernel_heights=(2,3,4)),
}

results = {}
for name, model in models.items():
    acc, models[name] = train_and_eval(model, train_loader, test_loader, epochs=40)
    bar = '█' * int(acc * 20)
    results[name] = acc
    print(f'  {name:<28} | {acc*100:5.1f}% | {bar}')

  Simple CNN1D (k=3)           |  50.0% | ██████████
  TextCNN 1D (k=2,3,4)         |  50.0% | ██████████
  TextCNN 2D (k=2,3,4)         |  33.3% | ██████


## 7️⃣ 필터 활성화 분석 & 새 문장 추론

In [8]:
tcnn = models['TextCNN 1D (k=2,3,4)'].to(device).eval()

# 필터 활성화
print('[필터 활성화 분석]')
phrases = ['amazing wonderful fantastic', 'terrible horrible worst',
           'great excellent brilliant',   'boring dull awful']
with torch.no_grad():
    for phrase in phrases:
        ids  = torch.tensor([vocab.encode(phrase, MAX_LEN)], device=device)
        emb  = tcnn.embedding(ids).permute(0, 2, 1)
        act  = F.relu(tcnn.convs[0](emb)).max().item()
        tag  = '긍정↑' if 'amazing' in phrase or 'great' in phrase else '부정↑'
        print(f"  '{phrase}' → 최대활성화: {act:.3f}  ({tag})")

# 새 문장 추론
print('\n[새 문장 감성 예측]')
test_sents = [
    'this is a great and wonderful film',
    'terrible and boring movie awful waste',
    'not bad actually quite enjoyable experience',
    'absolutely fantastic performance loved every scene',
]
with torch.no_grad():
    for sent in test_sents:
        ids  = torch.tensor([vocab.encode(sent, MAX_LEN)], device=device)
        prob = torch.softmax(tcnn(ids), dim=1)[0]
        pred = '긍정 😊' if prob[1] > 0.5 else '부정 😞'
        print(f"  '{sent}'")
        print(f"   → {pred}  (긍:{prob[1]:.2f} / 부:{prob[0]:.2f})\n")

[필터 활성화 분석]
  'amazing wonderful fantastic' → 최대활성화: 0.596  (긍정↑)
  'terrible horrible worst' → 최대활성화: 0.255  (부정↑)
  'great excellent brilliant' → 최대활성화: 0.439  (긍정↑)
  'boring dull awful' → 최대활성화: 0.401  (부정↑)

[새 문장 감성 예측]
  'this is a great and wonderful film'
   → 긍정 😊  (긍:0.89 / 부:0.11)

  'terrible and boring movie awful waste'
   → 부정 😞  (긍:0.08 / 부:0.92)

  'not bad actually quite enjoyable experience'
   → 부정 😞  (긍:0.21 / 부:0.79)

  'absolutely fantastic performance loved every scene'
   → 긍정 😊  (긍:0.73 / 부:0.27)



## 8️⃣ 실험: kernel_size 비교

다양한 `kernel_size` 조합의 성능을 비교합니다.

In [9]:
kernel_experiments = [
    (2, 3),
    (3, 4),
    (2, 3, 4),
    (2, 3, 4, 5),
    (1, 2, 3, 4, 5),
]

print('[Kernel Size 조합 실험]')
for ks in kernel_experiments:
    m = TextCNN(len(vocab), embed_dim=64, num_filters=32, kernel_sizes=ks)
    acc, _ = train_and_eval(m, train_loader, test_loader, epochs=40)
    bar = '█' * int(acc * 20)
    print(f'  kernels={str(ks):<15} | {acc*100:5.1f}% | {bar}')

[Kernel Size 조합 실험]
  kernels=(2, 3)          |  66.7% | █████████████
  kernels=(3, 4)          |  50.0% | ██████████
  kernels=(2, 3, 4)       |  66.7% | █████████████
  kernels=(2, 3, 4, 5)    |  50.0% | ██████████
  kernels=(1, 2, 3, 4, 5) |  66.7% | █████████████
